### **ACID - PART 3 - Control image quality**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2025/12/12


## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [17]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name
from image_quality_control.measure_plls import compute_plls
# from utils.mksubdir import mk_subdir
# from image_processing.extract_metadata import extract_bioio_scene_metadata
# from image_processing.name_metadata import extract_name_metadata
# from image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
# from utils.open_image import bioio_open_image
# from utils.save_image import tifffile_save_ometiff
# from image_processing.save_metadata import save_xml_string



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [18]:
# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"

# indicate the path to the directory storing the fields of view
fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov"

# # indicate the path to the directory where outputs will be saved
# output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\251205_train_test_split"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestap of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column="is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val=1

# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv"
default_metadata_file_exclude = None

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for power log-log slope calculation ---
channel_axis = 0


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# ignore indexes when saving pandas dataframes as csv files
save_csv_index = False # if False, the index will not be saved as a separate column in the csv file

# metadata saving date format
metadata_date_format = '%Y%m%d'

# metadata savingword
metadata_savingword = "metadata"

# metadata file suffix
metadata_file_suffix = f"part{save_file_name_separator}3.csv"

# hyperparameters saving date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}3.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



### Create secondary output directory if it doesn't exist - this directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [19]:
# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 2

Run the following cell.

Don't modify the following cell.

In [20]:

# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df



using 20251211_ACID_metadata_part_2.csv as default metadata file


#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [21]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), "Not all rows in metadata_df belong to the train set."

# Display the metadata dataframe
metadata_df

,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,infectious_organism,...,size_c,size_z,size_y,physical_size_y,size_x,physical_size_x,dims_order,int_well,treatment,is_train
0,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A1,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
4,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A5,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
5,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
7,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,B7,251211,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,1,uninfected_nocpd,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1166,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,F3,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_F3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1
1170,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G2,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1
1172,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G4,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1
1174,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G6,251211,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,DENV2,...,5,1,1024,0.325,1024,0.325,CYX,8,uninfected_JNJ_A07,1


#### Test PLLS computation

In [22]:

# initialize a global collection list
glob_plls_collection_list = []


# iterate through the rows of the metadata dataframe
for i in metadata_df.index:

    print("---------    ---------")
    
    # ---------
    # OPEN FIELD OF VIEW AND SEGMENTATION
    # ---------

    # get the field of view file
    field_of_view_file = metadata_df.loc[i, 'ome_tif_file_name']

    # process file only if file exists
    if os.path.exists(os.path.join(fov_directory, str(field_of_view_file))):
        
        print(f"working on {field_of_view_file}")

        # open the field of view
        field_of_view = tifffile.imread(os.path.join(fov_directory, field_of_view_file))

        print(field_of_view.shape)

        # calculate the power law log-log slope of the Fourier transformation of the field of view, per each channel
        fov_plls = compute_plls(image=field_of_view,
                                axis=channel_axis)

        print(fov_plls)
# #         # get the cytosol segmentation file - this is not anymore from part 1 notebook, but it is generated below
# #         cytosol_segmentation_file = metadata_df.loc[i, 'cytosol_segmentation']
# #         cytosol_segmentation = tifffile.imread(os.path.join(segmentation_directory, cytosol_segmentation_file))

#         # get the nucleus segmentation file and open the segmentation mask
#         nucleus_segmentation_file = metadata_df.loc[i, 'nucleus_segmentation']
#         nucleus_segmentation_i = tifffile.imread(os.path.join(segmentation_directory, nucleus_segmentation_file))
        
#         # get the cell segmentation file and open the segmentation mask
#         cell_segmentation_file = metadata_df.loc[i, 'cell_segmentation']
#         cell_segmentation_i = tifffile.imread(os.path.join(segmentation_directory, cell_segmentation_file))
        
#         # ---------
#         # PREPROCESSING CELL AND NUCLEUS MASKS - OBTAIN THE CYTOSOL SEGMENTATION MASK FROM THE PREPROCESSING RESULT

#         # 1) Channel axis is moved to position -1, for compatibility with skimage.measure.regionprops
#         # 2) Cells and nuclei with abnormal dimension (smaller than highpass_threshold or bigger than lowpass_threshold) are removed
#         # 3) Cells with no nuclei and nuclei with no cells are removed
#         # 4) Cells touching the image edges are removed
#         # 5) Nuclei are forced to be entirely contained into corresponding cell masks
#         # 6) Nuclei belonging to same cell are joined and assigned the same label value of the "mother" cell
#         # 7) Pixels having the min or max values of original image data type in any of the channels are discarded
#         # 8) The matching of labels between nuclei and corresponding cells is double-checked
#         # 9) Subtract preprocessed nucleus mask from preprocessed cell mask to obtain cytosol segmentation mask - NOTE: the label matchin is ensured during this process
#         # 10) Channel are, individually and independently, very mildly gaussian smoothed
#         # FINAL NOTE: if a background function is passed (per each channel), the function is substracted from the respective channel, before point 10 (gaussian smoothing)
#         # ---------

#         field_of_view, nucleus_segmentation, cell_segmentation, cytosol_segmentation = preprocess_image_and_label(field_of_view_i,
#                                                                                                                   nucleus_segmentation_i,
#                                                                                                                   cell_segmentation_i,
#                                                                                                                   nuc_area_highpass=nuc_area_highpass,
#                                                                                                                   nuc_area_lowpass=nuc_area_lowpass,
#                                                                                                                   cel_area_highpass=cel_area_highpass,
#                                                                                                                   cel_area_lowpass=cel_area_lowpass,
#                                                                                                                   channel_axis=channel_axis,
#                                                                                                                   sigma=sigma,
#                                                                                                                   gaussian_kwargs=gaussian_kwargs,
#                                                                                                                   original_dtype=original_dtype,
#                                                                                                                   background_function=background_function,
#                                                                                                                   background_dtype=background_dtype)
        
#         # print progress
#         print("Image preprocessing is done. Cytosol segmentation mask obtained")
        
#         # ---------
#         # SAVE CYTOSOL SEGMENTATION RESULTS
#         # ---------

#         # form cytosol segmentation saving name
#         cyt_save_name = f"{field_of_view_file.removesuffix(ome_suffix)}{separator}{cytosol_suffix}{ome_suffix}"

#         # form a metadata dictionary to be used for saving metadata in the segmentation masks
#         segmentation_metadata_dict = {'processing_date_yymmdd':datetime.datetime.now().strftime('%y%m%d')}
#         for clm in metadata_df.columns:
#             if clm in ['raw_file_name', 'scene_name', 'ome_tif_file_name', 'location', 'microscope', 'objective', 'donor',
#                        'transfection', 'stiffness', 'stimulation', 'time_of_stimulation',
#                        'physical_size_unit_x', 'physical_size_unit_y', 'physical_size_y', 'size_x',
#                        'physical_size_x', ]:
                
#                 segmentation_metadata_dict[clm]=metadata_df.loc[i,clm]

#         # change metadata dictionary to an ImageJ compatible format
#         imagej_segmentation_metadata_dict = imagej_compatible_metadata_dict(segmentation_metadata_dict)

#         # save the cytosol segmentation
#         tifffile_save_ometiff(os.path.join(segmentation_directory,cyt_save_name),
#                                     data=cytosol_segmentation,
#                                     imagej=True,
#                                     photometric="minisblack",
#                                     metadata=imagej_segmentation_metadata_dict)

#         print("Cytosol segmentation has been saved")
        
#         # save nucleus and cell segmentation if desired
#         if save_nuc_cell_segmentation:
            
#             # form nucleus and cell segmentation saving names
#             nuc_save_name = f"{nucleus_segmentation_file.removesuffix(ome_suffix)}{separator}{filter_nuc_suffix}{ome_suffix}"
#             cel_save_name = f"{cell_segmentation_file.removesuffix(ome_suffix)}{separator}{filter_cel_suffix}{ome_suffix}"


#             # save the nucleus and segmentation
#             tifffile_save_ometiff(os.path.join(segmentation_directory,nuc_save_name),
#                                         data=nucleus_segmentation,
#                                         imagej=True,
#                                         photometric="minisblack",
#                                         metadata=imagej_segmentation_metadata_dict)
            
#             # save the nucleus and segmentation
#             tifffile_save_ometiff(os.path.join(segmentation_directory,cel_save_name),
#                                         data=cell_segmentation,
#                                         imagej=True,
#                                         photometric="minisblack",
#                                         metadata=imagej_segmentation_metadata_dict)
            
#             print("Nucleus and cell segmentation has been saved - NOTE: THEIR NAMES ARE NOT UPDATED IN THE METADATA DATAFRAME")

#         # ---------
#         # MEASUREMENT EXTRACTION
#         # ---------

#         # extract measurements for the nucleus and cytosol segmentations

#         nucleus_measurements = pd.DataFrame(regionprops_table(nucleus_segmentation,
#                                                               intensity_image=field_of_view,
#                                                               properties=['label', 'area', 'centroid', 'intensity_mean', 'intensity_max', 'intensity_min']))

#         cytosol_measurements = pd.DataFrame(regionprops_table(cytosol_segmentation,
#                                                               intensity_image=field_of_view,
#                                                               properties=['label', 'area', 'centroid', 'intensity_mean', 'intensity_max', 'intensity_min']))

#         # rename columns to specify nuclei and cytosol measurements
#         nucleus_measurements.rename(columns={col:f"{nucleus_suffix}{separator}{col}" for col in nucleus_measurements.columns if col!='label'}, inplace=True)
#         cytosol_measurements.rename(columns={col:f"{cytosol_suffix}{separator}{col}" for col in cytosol_measurements.columns if col!='label'}, inplace=True)
                
#         # join initial nucleus and cytosol measurements
#         fov_glob_features = nucleus_measurements.merge(cytosol_measurements, on='label', how='outer')

#         # print progress
#         print("measurements: done")

#         # ---------
#         # CALCULATE BACKGROUND OFFSET PER CHANNEL - THIS IS DONE BY CALCULATING THE MEDIAN OF THE PIXELS NOT BELONGING TO ANY SEGMENTATION MASK
#         # ---------

#         # join cell and nucleus segmentation masks - NOTE: for this purpose, the original segmentation masks are used, not the preprocessed ones
#         # this is because the preprocessed masks have removed some cells/nuclei (e.g. those touching the image edge), in addition, the orignal
#         # segmentation have likely picked up artifacts, which are therefore excluded from the background calculation
        
#         background_mask = np.where(cell_segmentation_i>0,0,1)
#         background_mask = np.where(nucleus_segmentation_i>0,background_mask,1)
        
#         # iterate through the channels of the field of view - NOTE: channels are in position -1 from the preprocessing
#         for c in range(field_of_view.shape[-1]):
            
#             # get the channel array for the field of view
#             fov_ch = field_of_view[...,c]

#             # calculate the median of the pixels which are not in any segmentation mask
#             ch_background_offset = np.median(fov_ch[background_mask>0])
            
#             # add ch_backgound_offset to the measurement dataframe for the field of view
#             fov_glob_features[f'background_offset-{c}'] = [ch_background_offset for i in range(fov_glob_features.shape[0])]

#         # ---------
#         # ADD METADATA
#         # ---------
#         # get field_of_view metadata
#         fov_metadata = metadata_df[metadata_df['ome_tif_file_name']==field_of_view_file].copy()
        
#         # add cytosol segmentation metadata
#         fov_metadata[f'segmentation_date_yymmdd{separator}{cytosol_suffix}'] = datetime.datetime.now().strftime('%y%m%d')
#         fov_metadata['cytosol_segmentation'] = cyt_save_name
        
#         # transform metadata series into a dataframe with elements as columns, with the same number of rows as the
#         # glob_features dataframe and the values repeated over the rows
#         fov_metadata_df = pd.DataFrame({k: [fov_metadata[k].to_numpy()[0]] * fov_glob_features.shape[0] for k in fov_metadata})

#         # concatenate metadata and glob_features
#         fov_glob_features = pd.concat([fov_metadata_df, fov_glob_features], axis=1, ignore_index=False)

#         # print progress
#         print("add metadata: done")

#         # ---------
#         # COLLECT MEASUREMENTS IN GLOBAL COLLECTION LIST
#         # ---------
#         glob_features_collection_list.append(fov_glob_features)

# # concatenate measurements of all fields of view
# glob_features = pd.concat(glob_features_collection_list, axis=0, ignore_index=True)

# print("")
# print("=== === ===")
# print("feature extraction finished")

# # save the result
# glob_features_saving_name = f"{datetime.datetime.now().strftime('%y%m%d')}_nef_translocation_glob_measurements.csv"
# glob_features.to_csv(os.path.join(output_directory, glob_features_saving_name))

# # save results in an excel compatible format
# glob_features_saving_name_excl = f"{datetime.datetime.now().strftime('%y%m%d')}_nef_translocation_glob_measurements_excel.csv"
# glob_features.to_csv(os.path.join(output_directory, glob_features_saving_name_excl), sep=';', decimal=',', index=False)

# print("results saved")



---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B6.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B5.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B3.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B1.ome.tif
(5, 1024, 1024)
---------    ---------
working on H7_DENV2_MOI1_30h_fix

KeyboardInterrupt: 

#### Add train-test split column to metadata dataframe and save the results

Run the following cell.

Don't modify the following cell.

In [ ]:


# # save the metadata dataframe with train test split column
# # save metadata dataframe as a csv file
# metadata_saving_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
# metadata_df_w_split.to_csv(os.path.join(metadata_directory,metadata_saving_name), index=save_csv_index)

# # display metadata dataframe with train test split column
# metadata_df_w_split


### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# # # collect hyperparameters in a dictionary

# hyperparameter_dict = {

# 'test_size':test_size,
# 'metadata_directory':metadata_directory,
# 'plate_layout_directory':plate_layout_directory,
# 'output_directory':output_directory,
# 'metadata_file_name':metadata_file_name,
# 'plate_layout_file_name':plate_layout_file_name,
# 'is_train_column':is_train_column,
# 'train_val':train_val,
# 'test_val':test_val,
# 'train_test_split_kwargs':train_test_split_kwargs,
# 'concat_kwargs':concat_kwargs,
# 'default_metadata_file_target':default_metadata_file_target,
# 'default_metadata_file_exclude':default_metadata_file_exclude,
# 'default_platelayout_file_target':default_platelayout_file_target,
# 'default_platelayout_file_exclude':default_platelayout_file_exclude,
# 'metadata_from_file_name':metadata_from_file_name,
# 'platelayout_from_file_name':platelayout_from_file_name,
# 'metadata_default_separator':metadata_default_separator,
# 'platelayout_default_separator':platelayout_default_separator,
# 'metadata_default_date_position':metadata_default_date_position,
# 'platelayout_default_date_position':platelayout_default_date_position,
# 'metadata_default_date_format':metadata_default_date_format,
# 'platelayout_default_date_format':platelayout_default_date_format,
# 'metadata_default_reverse':metadata_default_reverse,
# 'platelayout_default_reverse':platelayout_default_reverse,
# 'save_file_name_separator':save_file_name_separator,
# 'project_name':project_name,
# 'metadata_date_format':metadata_date_format,
# 'metadata_savingword':metadata_savingword,
# 'metadata_file_suffix':metadata_file_suffix,
# 'hyperparameters_date_format':hyperparameters_date_format,
# 'hyperparameters_savingword':hyperparameters_savingword,
# 'hyperparameters_file_suffix':hyperparameters_file_suffix,
# 'secondary_output_directory':secondary_output_directory,
# 'exist_ok':exist_ok
# }



# # transform the hyperparameter_dict in a pandas series
# hyperparameter_series = pd.Series(hyperparameter_dict)

# # save hyperparamters
# hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
# hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name), index=save_csv_index)

